# Customer Churn Prediction — Initial Data Audit

**Phase 1 scope:** inspect the original customer data, record quality risks, and propose later cleaning decisions. This notebook does not clean data, engineer features, train models, or perform advanced EDA.

For every section, the notes explain what is checked, why it matters, and how to interpret the output.

## Data integrity rules

The source CSV is read only. No customer rows are changed or deleted. Missing values and statistical outliers are reported, not corrected. `customer_id` is an identifier and `churn` is the target; neither is treated as an ordinary predictor. Any preprocessing in a later phase must be learned after the train/test split to prevent leakage.

## 1. Imports and display settings

We import only the libraries required for a tabular audit. Fixed display settings make the outputs easier to review without changing the data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Load the original CSV

The path search starts at the current working directory and checks its parents. This makes the notebook reliable whether Jupyter starts from the repository root or from `notebooks/`. Loading the file without writing back to it preserves the raw source.

In [2]:
DATA_FILENAME = "customer_churn_advanced_dataset.csv"
candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / "data" / "raw" / DATA_FILENAME).is_file()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find data/raw/{DATA_FILENAME}. "
        "Place the original CSV in that location and rerun the notebook."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / DATA_FILENAME
df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: D:\Internship Tasks\Task - 2026.10.07\Customer-Churn-Prediction\data\raw\customer_churn_advanced_dataset.csv
Shape: 1,200 rows × 20 columns


## 3. First and last records

Viewing both ends checks that parsing produced sensible columns and values across the file. It can reveal shifted headers, delimiters, or trailing summary rows. The display is observational only.

In [ ]:
print("First five rows")
display(df.head())
print("Last five rows")
display(df.tail())

## 4. Shape, columns, data types, and feature roles

Data types determine which validation and preprocessing methods are appropriate. We separate predictors by analytical meaning: the numeric binary field `senior_citizen` is categorical in meaning. The identifier and target are explicitly excluded from predictor lists.

In [4]:
IDENTIFIER = "customer_id"
TARGET = "churn"
BINARY_CATEGORICAL = ["senior_citizen"]

column_overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_missing": df.notna().sum().values,
    "unique_non_missing": df.nunique(dropna=True).values,
})
display(column_overview)

numeric_features = [
    col for col in df.columns
    if pd.api.types.is_numeric_dtype(df[col])
    and col not in {IDENTIFIER, TARGET, *BINARY_CATEGORICAL}
]
categorical_features = [
    col for col in df.columns
    if col not in {IDENTIFIER, TARGET, *numeric_features}
]

print(f"Numerical predictors ({len(numeric_features)}): {numeric_features}")
print(f"Categorical/binary predictors ({len(categorical_features)}): {categorical_features}")
print(f"Identifier excluded from predictors: {IDENTIFIER}")
print(f"Target excluded from predictors: {TARGET}")

,column,dtype,non_missing,unique_non_missing
0,customer_id,str,1200,1200
1,age,int64,1200,58
2,tenure_months,int64,1200,72
3,contract_type,str,1200,3
4,internet_service,str,1200,3
5,region,str,1200,9
6,monthly_charges,float64,1170,1067
7,total_charges,float64,1176,1176
8,payment_method,str,1200,4
9,paperless_billing,str,1200,2


Numerical predictors (9): ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'complaints_last_12m', 'late_payments_last_12m', 'satisfaction_score', 'monthly_usage_gb', 'discount_pct']
Categorical/binary predictors (9): ['contract_type', 'internet_service', 'region', 'payment_method', 'paperless_billing', 'dependents', 'support_plan', 'tech_support', 'senior_citizen']
Identifier excluded from predictors: customer_id
Target excluded from predictors: churn


## 5. Missing values

Missingness can reduce usable information and introduce bias. Counts show the scale, while percentages make columns comparable. Row-level patterns show whether missing fields tend to occur together. No values are filled in this audit.

In [5]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
}).sort_values(["missing_count", "missing_pct"], ascending=False)
display(missing_summary)

rows_with_missing = df.isna().any(axis=1)
print(f"Rows with at least one missing value: {rows_with_missing.sum():,} ({rows_with_missing.mean():.2%})")
print(f"Total missing cells: {df.isna().sum().sum():,}")

missing_columns = missing_summary.index[missing_summary["missing_count"] > 0].tolist()
if missing_columns:
    print("Missing-value patterns (False = present, True = missing):")
    display(df[missing_columns].isna().value_counts().rename("row_count").to_frame())

,missing_count,missing_pct
satisfaction_score,48,4.00
monthly_usage_gb,36,3.00
monthly_charges,30,2.50
total_charges,24,2.00
customer_id,0,0.00
age,0,0.00
tenure_months,0,0.00
contract_type,0,0.00
internet_service,0,0.00
region,0,0.00


Rows with at least one missing value: 132 (11.00%)
Total missing cells: 138
Missing-value patterns (False = present, True = missing):


row_count
satisfaction_score monthly_usage_gb monthly_charges total_charges           
False              False            False           False               1068
True               False            False           False                 44
False              True             False           False                 35
                   False            True            False                 25
                                    False           True                  22
True               False            True            False                  3
                                    False           True                   1
False              True             True            False                  1
                   False            True            True                   1

## 6. Duplicate records and customer IDs

Complete duplicate rows may double-count customers. Repeated or missing IDs can also break customer-level tracking even when the rows differ. We report both checks separately and do not delete anything.

In [6]:
duplicate_summary = pd.Series({
    "completely_duplicated_rows": int(df.duplicated().sum()),
    "missing_customer_ids": int(df[IDENTIFIER].isna().sum()),
    "repeated_id_rows_including_all_occurrences": int(df[IDENTIFIER].duplicated(keep=False).sum()),
    "duplicate_ids_beyond_first_occurrence": int(df[IDENTIFIER].duplicated().sum()),
    "distinct_non_missing_customer_ids": int(df[IDENTIFIER].nunique(dropna=True)),
}, name="count")
display(duplicate_summary.to_frame())

id_format_ok = df[IDENTIFIER].astype("string").str.fullmatch(r"CUST\d{6}", na=False)
print(f"IDs outside the observed CUST + six-digit pattern: {(~id_format_ok).sum():,}")

,count
completely_duplicated_rows,0
missing_customer_ids,0
repeated_id_rows_including_all_occurrences,0
duplicate_ids_beyond_first_occurrence,0
distinct_non_missing_customer_ids,1200


IDs outside the observed CUST + six-digit pattern: 0


## 7. Target distribution

The class balance affects splitting, model evaluation, and retention targeting. Counts and percentages reveal whether accuracy could be misleading. The meaning of 0 and 1 must still be confirmed with the data owner.

In [7]:
target_counts = df[TARGET].value_counts(dropna=False).sort_index()
target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_counts.div(len(df)).mul(100),
})
display(target_distribution)
print(f"Observed target values: {sorted(df[TARGET].dropna().unique().tolist())}")

,count,percentage
churn,,
0,164,13.67
1,1036,86.33


Observed target values: [0, 1]


## 8. Descriptive statistics

Numerical summaries reveal ranges, centers, and spread. Categorical summaries show cardinality and the most common level. These results are a baseline for detecting unusual values, not a basis for changing them automatically.

In [8]:
print("Numerical descriptive statistics")
display(df.select_dtypes(include="number").describe().T)

print("Categorical descriptive statistics (customer_id excluded)")
display(df[categorical_features].astype("string").describe().T)

Numerical descriptive statistics


,count,mean,std,min,25%,50%,75%,max
age,"1,200.00",46.75,16.67,18.00,32.00,47.00,61.00,75.00
tenure_months,"1,200.00",36.27,20.92,1.00,18.00,35.00,54.00,72.00
monthly_charges,"1,170.00",71.43,25.82,20.00,54.45,71.82,87.00,235.01
total_charges,"1,176.00","2,568.01","1,807.26",22.73,"1,119.23","2,244.46","3,752.03","8,853.76"
complaints_last_12m,"1,200.00",0.71,0.85,0.00,0.00,1.00,1.00,5.00
late_payments_last_12m,"1,200.00",0.84,0.89,0.00,0.00,1.00,1.00,4.00
satisfaction_score,"1,152.00",6.74,1.78,1.00,5.60,6.75,8.00,10.00
monthly_usage_gb,"1,164.00",66.93,39.60,2.00,41.18,66.40,89.33,466.90
discount_pct,"1,200.00",9.16,6.61,0.00,3.58,8.71,13.70,32.89
senior_citizen,"1,200.00",0.17,0.38,0.00,0.00,0.00,0.00,1.00


Categorical descriptive statistics (customer_id excluded)


,count,unique,top,freq
contract_type,1200,3,Month-to-month,708
internet_service,1200,3,Fiber optic,621
region,1200,9,Western,147
payment_method,1200,4,Electronic check,392
paperless_billing,1200,2,Yes,784
dependents,1200,2,No,840
support_plan,1200,2,No,691
tech_support,1200,2,No,726
senior_citizen,1200,2,0,994


## 9. Category consistency

Blank strings, surrounding spaces, and inconsistent letter case can create false categories. We test for these formatting issues and list observed levels so they can be compared with a future data dictionary.

In [9]:
text_features = [col for col in categorical_features if col != "senior_citizen"]
category_checks = []
for col in text_features:
    values = df[col].dropna().astype("string")
    category_checks.append({
        "column": col,
        "unique_values": values.nunique(),
        "blank_strings": values.str.strip().eq("").sum(),
        "surrounding_whitespace": values.ne(values.str.strip()).sum(),
        "case_only_collisions": values.nunique() - values.str.strip().str.casefold().nunique(),
    })
display(pd.DataFrame(category_checks).set_index("column"))

for col in categorical_features:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist(), key=str)}")

,unique_values,blank_strings,surrounding_whitespace,case_only_collisions
column,,,,
contract_type,3,0,0,0
internet_service,3,0,0,0
region,9,0,0,0
payment_method,4,0,0,0
paperless_billing,2,0,0,0
dependents,2,0,0,0
support_plan,2,0,0,0
tech_support,2,0,0,0


contract_type: ['Month-to-month', 'One year', 'Two year']
internet_service: ['DSL', 'Fiber optic', 'No']
region: ['Central', 'Eastern', 'North Central', 'North Western', 'Northern', 'Sabaragamuwa', 'Southern', 'Uva', 'Western']
payment_method: ['Bank transfer', 'Credit card', 'Electronic check', 'Mailed check']
paperless_billing: ['No', 'Yes']
dependents: ['No', 'Yes']
support_plan: ['No', 'Yes']
tech_support: ['No', 'Yes']
senior_citizen: [0, 1]


## 10. Broad numerical validity screens

These deliberately broad ranges catch clearly impossible values while avoiding unsupported assumptions. For example, satisfaction is screened from 0 to 10 and percentages from 0 to 100. A zero count means only that the field passed this provisional screen; documented business limits may be narrower.

In [10]:
screening_rules = {
    "age": {"minimum": 18, "maximum": 100},
    "tenure_months": {"minimum": 0, "maximum": None},
    "monthly_charges": {"minimum": 0, "maximum": None},
    "total_charges": {"minimum": 0, "maximum": None},
    "complaints_last_12m": {"minimum": 0, "maximum": None},
    "late_payments_last_12m": {"minimum": 0, "maximum": None},
    "satisfaction_score": {"minimum": 0, "maximum": 10},
    "monthly_usage_gb": {"minimum": 0, "maximum": None},
    "discount_pct": {"minimum": 0, "maximum": 100},
}

validity_results = []
for col, rule in screening_rules.items():
    non_missing = df[col].notna()
    invalid = pd.Series(False, index=df.index)
    if rule["minimum"] is not None:
        invalid |= non_missing & df[col].lt(rule["minimum"])
    if rule["maximum"] is not None:
        invalid |= non_missing & df[col].gt(rule["maximum"])
    validity_results.append({
        "column": col,
        "screen_min": rule["minimum"],
        "screen_max": rule["maximum"],
        "observed_min": df[col].min(),
        "observed_max": df[col].max(),
        "outside_screen": int(invalid.sum()),
    })

display(pd.DataFrame(validity_results).set_index("column"))
print("Non-binary senior_citizen values:", int((df["senior_citizen"].notna() & ~df["senior_citizen"].isin([0, 1])).sum()))
print("Non-binary churn values:", int((df[TARGET].notna() & ~df[TARGET].isin([0, 1])).sum()))

,screen_min,screen_max,observed_min,observed_max,outside_screen
column,,,,,
age,18,100.00,18.00,75.00,0
tenure_months,0,NaN,1.00,72.00,0
monthly_charges,0,NaN,20.00,235.01,0
total_charges,0,NaN,22.73,"8,853.76",0
complaints_last_12m,0,NaN,0.00,5.00,0
late_payments_last_12m,0,NaN,0.00,4.00,0
satisfaction_score,0,10.00,1.00,10.00,0
monthly_usage_gb,0,NaN,2.00,466.90,0
discount_pct,0,100.00,0.00,32.89,0


Non-binary senior_citizen values: 0
Non-binary churn values: 0


## 11. Potential outliers using the IQR rule

For each numerical predictor, values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR are flagged. This common method is a screening tool: an outlier may be a valid high-value customer or rare event. Binary fields, the target, and the identifier are excluded. No values are removed or capped.

In [11]:
outlier_flags = pd.DataFrame(False, index=df.index, columns=numeric_features)
outlier_rows = []

for col in numeric_features:
    observed = df[col].dropna()
    q1, q3 = observed.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flags = df[col].lt(lower) | df[col].gt(upper)
    outlier_flags[col] = flags
    flagged_values = df.loc[flags, col]
    outlier_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "lower_bound": lower,
        "upper_bound": upper,
        "flagged_count": int(flags.sum()),
        "flagged_pct": flags.mean() * 100,
        "flagged_min": flagged_values.min() if flags.any() else np.nan,
        "flagged_max": flagged_values.max() if flags.any() else np.nan,
    })

outlier_summary = pd.DataFrame(outlier_rows).set_index("column")
display(outlier_summary)
print(f"Rows flagged in at least one numerical field: {outlier_flags.any(axis=1).sum():,}")
print(f"Total field-level IQR flags: {outlier_flags.sum().sum():,}")

,q1,q3,lower_bound,upper_bound,flagged_count,flagged_pct,flagged_min,flagged_max
column,,,,,,,,
age,32.00,61.00,-11.50,104.50,0,0.00,NaN,NaN
tenure_months,18.00,54.00,-36.00,108.00,0,0.00,NaN,NaN
monthly_charges,54.45,87.00,5.62,135.82,12,1.00,136.90,235.01
total_charges,"1,119.23","3,752.03","-2,829.97","7,701.22",14,1.17,"7,704.76","8,853.76"
complaints_last_12m,0.00,1.00,-1.50,2.50,41,3.42,3.00,5.00
late_payments_last_12m,0.00,1.00,-1.50,2.50,57,4.75,3.00,4.00
satisfaction_score,5.60,8.00,2.00,11.60,4,0.33,1.00,1.90
monthly_usage_gb,41.18,89.33,-31.05,161.55,5,0.42,169.80,466.90
discount_pct,3.58,13.70,-11.60,28.87,2,0.17,29.76,32.89


Rows flagged in at least one numerical field: 125
Total field-level IQR flags: 135


## 12. Related-field consistency questions

Related fields can reveal definition or capture problems. The checks below are questions for the data owner, not asserted rules. Charges can vary over time, discounts can apply, and service/support fields may cover different products. We therefore flag large differences and surprising combinations without editing them.

In [ ]:
expected_from_current_rate = df["monthly_charges"] * df["tenure_months"]
charge_ratio = df["total_charges"] / expected_from_current_rate.replace(0, np.nan)
relative_charge_difference = (df["total_charges"] - expected_from_current_rate).abs() / expected_from_current_rate.replace(0, np.nan)
large_charge_difference = relative_charge_difference.gt(0.50)

print(f"Rows with total_charges differing by more than 50% from current monthly charge × tenure: {large_charge_difference.sum():,}")
charge_review = df.loc[large_charge_difference, [IDENTIFIER, "tenure_months", "monthly_charges", "total_charges"]].copy()
charge_review["total_to_current_rate_ratio"] = charge_ratio[large_charge_difference]
display(charge_review)

relationship_questions = pd.Series({
    "internet_service_No_and_positive_usage": int(((df["internet_service"] == "No") & df["monthly_usage_gb"].gt(0)).sum()),
    "internet_service_No_and_tech_support_Yes": int(((df["internet_service"] == "No") & (df["tech_support"] == "Yes")).sum()),
    "tech_support_Yes_and_support_plan_No": int(((df["tech_support"] == "Yes") & (df["support_plan"] == "No")).sum()),
    "tech_support_No_and_support_plan_Yes": int(((df["tech_support"] == "No") & (df["support_plan"] == "Yes")).sum()),
}, name="records_to_clarify")
display(relationship_questions.to_frame())
print("Note: senior_citizen is not recalculated from age because its defining threshold is not documented.")

## 13. Concise audit summary and proposed decisions

This final section brings the main findings together. The proposed decisions are for Phase 2 review; this notebook does not apply them.

In [13]:
audit_summary = pd.DataFrame([
    {"area": "Dataset", "finding": f"{len(df):,} rows and {df.shape[1]} columns", "risk_or_decision": "Preserve the raw CSV unchanged."},
    {"area": "Missingness", "finding": f"{rows_with_missing.sum():,} rows ({rows_with_missing.mean():.2%}); {df.isna().sum().sum():,} cells", "risk_or_decision": "Investigate cause; fit any imputer on training data only."},
    {"area": "Duplicates", "finding": f"{df.duplicated().sum():,} full duplicates; {df[IDENTIFIER].duplicated().sum():,} repeated IDs", "risk_or_decision": "No row removal indicated by duplicate checks."},
    {"area": "Target", "finding": f"Class 1: {target_counts.get(1, 0):,} ({target_counts.get(1, 0) / len(df):.2%})", "risk_or_decision": "Confirm label meaning; use stratification and imbalance-aware metrics."},
    {"area": "Outliers", "finding": f"{outlier_flags.any(axis=1).sum():,} rows flagged by 1.5×IQR", "risk_or_decision": "Verify and retain initially; do not remove automatically."},
    {"area": "Cross-field checks", "finding": f"{large_charge_difference.sum():,} large charge differences; service/usage questions present", "risk_or_decision": "Ask the data owner before defining a correction rule."},
    {"area": "Leakage prevention", "finding": "customer_id is unique and excluded from predictors", "risk_or_decision": "Split before imputation, encoding, scaling, or threshold selection."},
])
display(audit_summary)

print("Proposed Phase 2 sequence:")
print("1. Confirm target semantics, field definitions, units, and allowed ranges.")
print("2. Make a stratified split while retaining customer_id separately for traceability.")
print("3. Investigate missingness and fit approved imputation only on training data.")
print("4. Verify flagged charge/service combinations and IQR extremes against source records.")
print("5. Put encoding and any scaling or robust treatment inside a fitted training pipeline.")

,area,finding,risk_or_decision
0,Dataset,"1,200 rows and 20 columns",Preserve the raw CSV unchanged.
1,Missingness,132 rows (11.00%); 138 cells,Investigate cause; fit any imputer on training...
2,Duplicates,0 full duplicates; 0 repeated IDs,No row removal indicated by duplicate checks.
3,Target,"Class 1: 1,036 (86.33%)",Confirm label meaning; use stratification and ...
4,Outliers,125 rows flagged by 1.5×IQR,Verify and retain initially; do not remove aut...
5,Cross-field checks,5 large charge differences; service/usage ques...,Ask the data owner before defining a correctio...
6,Leakage prevention,customer_id is unique and excluded from predic...,"Split before imputation, encoding, scaling, or..."


Proposed Phase 2 sequence:
1. Confirm target semantics, field definitions, units, and allowed ranges.
2. Make a stratified split while retaining customer_id separately for traceability.
3. Investigate missingness and fit approved imputation only on training data.
4. Verify flagged charge/service combinations and IQR extremes against source records.
5. Put encoding and any scaling or robust treatment inside a fitted training pipeline.


## Conclusion

The dataset is structurally usable, but its high churn prevalence, missing values in four fields, extreme financial/usage observations, and several cross-field questions require review. Passing a broad range screen does not prove that a value is correct. The detailed findings and proposed leakage-safe cleaning approach are recorded in `reports/data_audit.md`. **Phase 1 stops here.**